# Exercise 3: Text-based Software Analysis (2)

In GitHub Codespaces, select the Python kernel at `/opt/jupyter/bin/python`. Run the cells in order.

**This notebook is companion** to exercise sheet `asa-exercise-03-regex-strengths-and-limitations.pdf`. Look there for context and instructions.

## Flexibility

The sheet gives a C program with an SPDX license header and a Python program with a syntax error.

In [ ]:
import re

c_source = """// SPDX-License-Identifier: Apache-2.0
int main() { }
"""

c_spdx_pattern = r"// SPDX-License-Identifier: .+"

print("Match" if re.search(c_spdx_pattern, c_source) else "No match")

Adjust the expression below to find the SPDX license header in the Python program. Then identify its syntax error and explain on the sheet why the analysis still works.

In [ ]:
python_source = """# SPDX-License-Identifier: Apache-2.0
def greet:
    print("Hello")
"""

python_spdx_pattern = r"// SPDX-License-Identifier: .+"
print("Match" if re.search(c_spdx_pattern, python_source) else "No match")

## Source Positions

Extend the `for match in re.finditer(...)` loop in `check_source` to report each match's start and end in the given string, and the matched text. The sample has two matches on line 2 and one match on line 3.

See [the documentation of re.Match](https://docs.python.org/3/library/re.html#match-objects) to find the required method for getting the end of a match.

Expected output:

```
Violation from 31--38: the the
Violation from 51--58: who who
Violation from 82--91: each each
```

In [ ]:
def check_source(pattern: str, source: str):
    has_match = False
    for match in re.finditer(pattern, string=source):
        has_match = True
        print(f"Violation at {match.start()}: {match.group(0)}")
    if not has_match:
        print("No violation found")


repeated_words_source = """int main(void) {
    // Review the the input; note who who owns it.
    // Handle each each result.
    return 0;
}
"""
repeated_words_pattern = r"\b([A-Za-z]+)[ \t]+\1\b"

check_source(pattern=repeated_words_pattern, source=repeated_words_source)

## Limitations

For each fixed expression on the sheet, provide one false-positive string and one false-negative string. A short string is enough; it need not be a complete or compilable program. Test both strings with `check_source` from the source-position task.

_Reminder: false-positive is a violation found even though there shouldn't be one. false-negative is no violation found even though there should be one._

### Property 0: There is no call to function `exit`

The expression from the sheet is `r"\bexit\s*\("`.

In [ ]:
exit_pattern = r"\bexit\s*\("

exit_false_positive = ""
exit_false_negative = ""

print("False-positive candidate:")
check_source(exit_pattern, exit_false_positive)
print("False-negative candidate:")
check_source(exit_pattern, exit_false_negative)

### Property 1: There is no SPDX license identifier comment

Use the same C license-header expression as in the flexibility task: `r"// SPDX-License-Identifier: .+"`.

In [ ]:
c_spdx_pattern = r"// SPDX-License-Identifier: .+"

spdx_false_positive = ""
spdx_false_negative = ""

print("False-positive candidate:")
check_source(c_spdx_pattern, spdx_false_positive)
print("False-negative candidate:")
check_source(c_spdx_pattern, spdx_false_negative)

### Property 2: There is no word accidentally repeated in succession

Use the same repeated-word expression as in the source-position task: `r"\b([A-Za-z]+)[ \t]+\1\b"`.

In [ ]:
repeated_words_pattern = r"\b([A-Za-z]+)[ \t]+\1\b"

repeated_words_false_positive = ""
repeated_words_false_negative = ""

print("False-positive candidate:")
check_source(repeated_words_pattern, repeated_words_false_positive)
print("False-negative candidate:")
check_source(repeated_words_pattern, repeated_words_false_negative)

## For Enthusiasts: Diagnostic Output

Extend `check_source` from the source-position task. For every match, print a warning, the file name, its line and starting column, the source line, and a caret under each character of the match. Aim for this output on the input below:

```text
Warning: repeated words detected: `the the`
  --> file src/example.c:2:3
   |
 2 | # the the solution is as expected,
   |   ^^^^^^^
   |
   = hint: remove one of the repeated words.
```

`src/example.c:2:3` means file `src/example.c`, line 2, column 3. This is the location of the start of the found match.
Below, the output shows the full line of the match, and the carets `^^^^` below the line mark the full match (from start to end).

Your function needs the file name and a warning message as additional inputs. One possible signature is:

```python
def check_source(pattern: str, code: str, warning_msg: str, file_name: str) -> bool:
```

Scan the source one line at a time with `for line_count, line in enumerate(code.splitlines(), start=1):`. Use `re.finditer(pattern, line)` to find every occurrence on that line. A match's `start()` is the offset within the line, starting at 0; use it to compute the column and align the carets.

In [ ]:
example_code = """int main(void) {
# the the solution is as expected,
}
"""
warning_msg = "repeated words detected"
file_name = "src/example.c"